# Deep Past Initiative – Machine Translation (Inference Notebook)

This notebook is a **starter / baseline** for this Kaggle competition.

Training Code is [here](https://www.kaggle.com/code/takamichitoda/dpc-starter-train).

In [ ]:
import os
import re
import math
import unicodedata
from typing import List

import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import ByT5Tokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm

# =========================
# CONFIG
# =========================
MODEL_PATH = "/kaggle/input/datasets/lihu6666/36-5modelbyt5/byt5-large-akkadian-model-v2"
TEST_DATA_PATH = "/kaggle/input/deep-past-initiative-machine-translation/test.csv"

BATCH_SIZE = 16
MAX_LENGTH = 512
MAX_NEW_TOKENS = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PREFIX = "translate Akkadian to English: "
FALLBACK_TEXT = "The tablet contains fragmentary text."

print(f"DEVICE={DEVICE}")
print(f"MODEL_PATH={MODEL_PATH}")
print(f"TEST_DATA_PATH={TEST_DATA_PATH}")

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model path not found: {MODEL_PATH}")
if not os.path.exists(TEST_DATA_PATH):
    raise FileNotFoundError(f"Test file not found: {TEST_DATA_PATH}")

print("Model dir files:", os.listdir(MODEL_PATH)[:30])

# =========================
# INPUT PREPROCESS
# =========================
_V2 = re.compile(r"([aAeEiIuU])(?:2|₂)")
_V3 = re.compile(r"([aAeEiIuU])(?:3|₃)")
_ACUTE = str.maketrans({
    "a": "á", "e": "é", "i": "í", "u": "ú",
    "A": "Á", "E": "É", "I": "Í", "U": "Ú"
})
_GRAVE = str.maketrans({
    "a": "à", "e": "è", "i": "ì", "u": "ù",
    "A": "À", "E": "È", "I": "Ì", "U": "Ù"
})

def _ascii_to_diacritics(s: str) -> str:
    s = s.replace("sz", "š").replace("SZ", "Š")
    s = s.replace("s,", "ṣ").replace("S,", "Ṣ")
    s = s.replace("t,", "ṭ").replace("T,", "Ṭ")
    s = _V2.sub(lambda m: m.group(1).translate(_ACUTE), s)
    s = _V3.sub(lambda m: m.group(1).translate(_GRAVE), s)
    return s

_ALLOWED_FRACS = [
    (1/6, "0.16666"),
    (1/4, "0.25"),
    (1/3, "0.33333"),
    (1/2, "0.5"),
    (2/3, "0.66666"),
    (3/4, "0.75"),
    (5/6, "0.83333"),
]
_FRAC_TOL = 2e-3
_FLOAT_RE = re.compile(r"(?<![\w/])(\d+\.\d{4,})(?![\w/])")

def _canon_decimal(x: float) -> str:
    ip = int(math.floor(x + 1e-12))
    frac = x - ip
    best = min(_ALLOWED_FRACS, key=lambda t: abs(frac - t[0]))

    if abs(frac - best[0]) <= _FRAC_TOL:
        dec = best[1]
        if ip == 0:
            return dec
        return f"{ip}{dec[1:]}" if dec.startswith("0.") else f"{ip}+{dec}"

    return f"{x:.5f}".rstrip("0").rstrip(".")

_TAG_BIGGAP_RE = re.compile(r"<\s*big[\s_\-]*gap\s*>", re.I)
_TAG_GAP_RE = re.compile(r"<\s*gap\s*>", re.I)
_BARE_BIGGAP = re.compile(r"\bbig[\s_\-]*gap\b", re.I)
_ELLIPSIS_RE = re.compile(r"(?:\.{3,}|…+|\[\.+\])")
_BRACKET_X_RE = re.compile(r"(\[\s*x\s*\]|\(\s*x\s*\))", re.I)
_XTOKEN_RUN_RE = re.compile(r"\bx(?:\s+x)+\b", re.I)
_XRUN_RE = re.compile(r"(?<!\w)x{2,}(?!\w)", re.I)
_XTOK_RE = re.compile(r"(?<!\w)x(?!\w)", re.I)
_WS_RE = re.compile(r"\s+")

_GAP_UNIFIED_RE = re.compile(
    r"<\s*big[\s_\-]*gap\s*>"
    r"|<\s*gap\s*>"
    r"|\bbig[\s_\-]*gap\b"
    r"|\bx(?:\s+x)+\b"
    r"|\.{3,}|…+|\[\.+\]"
    r"|\[\s*x\s*\]|\(\s*x\s*\)"
    r"|(?<!\w)x{2,}(?!\w)"
    r"|(?<!\w)x(?!\w)"
    r"|\(\s*large\s+break\s*\)"
    r"|\(\s*break\s*\)"
    r"|\(\s*\d+\s+broken\s+lines?\s*\)",
    re.I
)

def _normalize_gaps_vec(ser: pd.Series) -> pd.Series:
    return ser.str.replace(_GAP_UNIFIED_RE, "<gap>", regex=True)

_CHAR_TRANS = str.maketrans({
    "ḫ": "h", "Ḫ": "H", "ʾ": "",
    "₀": "0", "₁": "1", "₂": "2", "₃": "3", "₄": "4",
    "₅": "5", "₆": "6", "₇": "7", "₈": "8", "₉": "9",
    "—": "-", "–": "-",
})
_SUB_X = "ₓ"

_UNICODE_UPPER = r"A-ZŠṬṢḪ\u00C0-\u00D6\u00D8-\u00DE\u0160\u1E00-\u1EFF"
_UNICODE_LOWER = r"a-zšṭṣḫ\u00E0-\u00F6\u00F8-\u00FF\u0161\u1E01-\u1EFF"

_DET_UPPER_RE = re.compile(r"\(([" + _UNICODE_UPPER + r"0-9]{1,6})\)")
_DET_LOWER_RE = re.compile(r"\(([" + _UNICODE_LOWER + r"]{1,4})\)")

_PN_RE = re.compile(r"\bPN\b")
_KUBABBAR_RE = re.compile(r"KÙ\.B\.")
_EXACT_FRAC_RE = re.compile(r"0\.8333|0\.6666|0\.3333|0\.1666|0\.625|0\.75|0\.25|0\.5")
_EXACT_FRAC_MAP = {
    "0.8333": "⅚",
    "0.6666": "⅔",
    "0.3333": "⅓",
    "0.1666": "⅙",
    "0.625": "⅝",
    "0.75": "¾",
    "0.25": "¼",
    "0.5": "½",
}

def _frac_repl(m: re.Match) -> str:
    return _EXACT_FRAC_MAP[m.group(0)]

class OptimizedPreprocessor:
    def preprocess_batch(self, texts: List[str]) -> List[str]:
        ser = pd.Series(texts).fillna("").astype(str)

        # 先做 Unicode 规范化，避免奇怪字符形式不一致
        ser = ser.apply(lambda x: unicodedata.normalize("NFC", x))

        # ASCII -> 带附加符号转写
        ser = ser.apply(_ascii_to_diacritics)

        # 大写 determinative 直接去括号；小写 determinative 改成 {x}
        ser = ser.str.replace(_DET_UPPER_RE, r"\1", regex=True)
        ser = ser.str.replace(_DET_LOWER_RE, r"{\1}", regex=True)

        # 统一 gap
        ser = _normalize_gaps_vec(ser)

        # 字符级归一
        ser = ser.str.translate(_CHAR_TRANS)
        ser = ser.str.replace(_SUB_X, "", regex=False)

        # 特殊模式归一
        ser = ser.str.replace(_KUBABBAR_RE, "KÙ.BABBAR", regex=True)

        # 分数归一
        ser = ser.str.replace(_EXACT_FRAC_RE, _frac_repl, regex=True)
        ser = ser.str.replace(_FLOAT_RE, lambda m: _canon_decimal(float(m.group(1))), regex=True)

        # 可选：保留 PN，不额外改动
        # ser = ser.str.replace(_PN_RE, "PN", regex=True)

        # 空白压缩
        ser = ser.str.replace(_WS_RE, " ", regex=True).str.strip()

        return ser.tolist()

# =========================
# Conservative output postprocess
# =========================
_OUT_WS_RE = re.compile(r"\s+")
_CONTROL_CHAR_RE = re.compile(r"[\x00-\x08\x0b-\x1f]")

_OUT_TAG_BIGGAP_RE = re.compile(r"<\s*big[\s_\-]*gap\s*>", re.I)
_OUT_TAG_GAP_RE = re.compile(r"<\s*gap\s*>", re.I)
_OUT_GAP_LEGACY_RE = re.compile(r"(\[x\]|\(x\)|\bx\b)", re.I)
_OUT_ELLIPSIS_RE = re.compile(r"(?:\.{3,}|…+|……|\[\.+\])")
_OUT_PAREN_BREAK_RE = re.compile(r"\(\s*(?:large\s+break|break|\d+\s+broken\s+lines?)\s*\)", re.I)
_OUT_DUP_GAP_RE = re.compile(r"(?:<gap>\s*){2,}", re.I)

_GRAM_PARENS_RE = re.compile(
    r"\(\s*(?:fem|plur|pl|sing|singular|plural|\?|\!)"
    r"(?:\.\s*(?:plur|plural|sing|singular|pl|fem))?"
    r"\.?\s*[^)]*\)",
    re.I,
)
_EDITORIAL_TOKEN_RE = re.compile(
    r"(?<!\w)(?:fem\.|sing\.|pl\.|plural|\(\?\))(?!\w)",
    re.I,
)

_SQUARE_BRACKET_CONTENT_RE = re.compile(r"\[([A-Za-z][^\[\]<>]{0,80}?)\]")
_REPEAT_WORD_RE = re.compile(r"\b(\w+)(?:\s+\1\b)+", re.I)
_REPEAT_BIGRAM_NEAR_RE = re.compile(r"\b(\w+\s+\w+)(?:,\s*|\s+)\1\b", re.I)
_REPEAT_UNIGRAM_NEAR_RE = re.compile(r"\b(\w+)(?:,\s*|\s+)\1\b", re.I)

_SLASH_FRAC_RE = re.compile(r"(?<!\d)(1|2|5)\s*/\s*(2|3|4|6)(?!\d)")
_MIXED_INLINE_FRAC_RE = re.compile(r"(?<!\d)(\d+)([½¼⅓⅔⅙⅚])(?!\d)")

_DECIMAL_FRAC_PATTERNS = [
    (re.compile(r"(?<!\d)0?\.5(?!\d)"), "½"),
    (re.compile(r"(?<!\d)0?\.50(?!\d)"), "½"),
    (re.compile(r"(?<!\d)0?\.25(?!\d)"), "¼"),
    (re.compile(r"(?<!\d)0?\.250(?!\d)"), "¼"),
    (re.compile(r"(?<!\d)0?\.(?:3{2,}|33)(?!\d)"), "⅓"),
    (re.compile(r"(?<!\d)0?\.(?:6{2,}|67|66)(?!\d)"), "⅔"),
    (re.compile(r"(?<!\d)0?\.(?:1(?:6+|7)|16)(?!\d)"), "⅙"),
    (re.compile(r"(?<!\d)0?\.(?:8(?:3+|4)|83)(?!\d)"), "⅚"),
]

UNICODE_FRAC_MAP = {
    ("1", "2"): "½",
    ("1", "4"): "¼",
    ("1", "3"): "⅓",
    ("2", "3"): "⅔",
    ("1", "6"): "⅙",
    ("5", "6"): "⅚",
}

def slash_fraction_to_unicode(num: str, den: str) -> str:
    return UNICODE_FRAC_MAP.get((num, den), f"{num}/{den}")

def conservative_postprocess(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return FALLBACK_TEXT

    t = unicodedata.normalize("NFC", text)
    t = _CONTROL_CHAR_RE.sub("", t)

    t = _OUT_TAG_BIGGAP_RE.sub("<gap>", t)
    t = _OUT_TAG_GAP_RE.sub("<gap>", t)
    t = _OUT_GAP_LEGACY_RE.sub("<gap>", t)
    t = _OUT_ELLIPSIS_RE.sub("<gap>", t)
    t = _OUT_PAREN_BREAK_RE.sub("<gap>", t)

    t = _GRAM_PARENS_RE.sub(" ", t)
    t = _EDITORIAL_TOKEN_RE.sub(" ", t)
    t = _SQUARE_BRACKET_CONTENT_RE.sub(r"\1", t)

    def _slash_repl(m):
        return slash_fraction_to_unicode(m.group(1), m.group(2))

    t = _SLASH_FRAC_RE.sub(_slash_repl, t)

    for pat, repl in _DECIMAL_FRAC_PATTERNS:
        t = pat.sub(repl, t)

    t = _MIXED_INLINE_FRAC_RE.sub(r"\1 \2", t)

    t = t.replace("<gap>", "\x00GAP\x00")
    t = _OUT_WS_RE.sub(" ", t).strip()
    t = t.replace("\x00GAP\x00", " <gap> ")

    t = re.sub(r"(?<=-)\s+<gap>\s+", "<gap>", t)
    t = re.sub(r"\s+<gap>(?=-)", "<gap>", t)
    t = _OUT_DUP_GAP_RE.sub("<gap> ", t)

    for _ in range(2):
        t = _REPEAT_BIGRAM_NEAR_RE.sub(r"\1", t)
        t = _REPEAT_UNIGRAM_NEAR_RE.sub(r"\1", t)
        t = _REPEAT_WORD_RE.sub(r"\1", t)

    for n in range(4, 1, -1):
        pat = r"\b((?:\w+\s+){" + str(n - 1) + r"}\w+)(?:\s+\1\b)+"
        t = re.sub(pat, r"\1", t, flags=re.I)

    t = re.sub(r"\s+([,.;:!?])", r"\1", t)
    t = re.sub(r"([,.;:!?])([A-Za-z])", r"\1 \2", t)
    t = _OUT_WS_RE.sub(" ", t).strip().strip("-").strip()

    if not t:
        return FALLBACK_TEXT

    if t[0].islower():
        t = t[0].upper() + t[1:]
    if t[-1] not in ".!?":
        t += "."

    return t

# =========================
# Model Loading
# =========================
print("Loading tokenizer...")
tokenizer = ByT5Tokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
)

print(f"Loading model from {MODEL_PATH}...")
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
).to(DEVICE)
model.eval()

# =========================
# Dataset
# =========================
test_df = pd.read_csv(TEST_DATA_PATH)

if "id" not in test_df.columns or "transliteration" not in test_df.columns:
    raise ValueError(f"Unexpected test columns: {list(test_df.columns)}")

preprocessor = OptimizedPreprocessor()

class InferenceDataset(Dataset):
    def __init__(self, df, tokenizer, preprocessor):
        raw_texts = df["transliteration"].fillna("").astype(str).tolist()
        proc_texts = preprocessor.preprocess_batch(raw_texts)

        self.texts = [PREFIX + x for x in proc_texts]
        self.tokenizer = tokenizer

        # 调试时可以看看前几个样本
        print("\nSample preprocessed inputs:")
        for i in range(min(3, len(raw_texts))):
            print(f"[RAW {i}] {raw_texts[i]}")
            print(f"[PP  {i}] {proc_texts[i]}")
            print("-" * 80)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        inputs = self.tokenizer(
            text,
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
        }

test_dataset = InferenceDataset(test_df, tokenizer, preprocessor)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

# =========================
# Inference
# =========================
print("Starting inference...")
all_predictions = []

with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            num_beams=4,
            early_stopping=True,
            max_new_tokens=MAX_NEW_TOKENS,
            use_cache=True,
            do_sample=False,
        )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        decoded = [conservative_postprocess(d) for d in decoded]
        all_predictions.extend(decoded)

# =========================
# Submission
# =========================
submission = pd.DataFrame({
    "id": test_df["id"],
    "translation": all_predictions
})

submission["translation"] = submission["translation"].apply(
    lambda x: x if isinstance(x, str) and x.strip() else FALLBACK_TEXT
)

submission.to_csv("submission.csv", index=False)
print("Submission file saved successfully!")
print(submission.head())
print("Avg len:", submission["translation"].astype(str).str.len().mean())
print("Empty predictions:", (submission["translation"].astype(str).str.strip() == "").sum())